In [1]:
import requests
import pandas as pd
from datetime import date

# ============================================
# District coordinates - Maharashtra
# ============================================
locations = {
    "Beed": {
        "latitude": 18.9891,
        "longitude": 75.7601
    },
    "Latur": {
        "latitude": 18.4088,
        "longitude": 76.5604
    },
    "Osmanabad": {
        "latitude": 18.1860,
        "longitude": 76.0419
    },
    "Solapur": {
        "latitude": 17.6599,
        "longitude": 75.9064
    },
    "Ahmednagar": {
        "latitude": 19.0948,
        "longitude": 74.7480
    }
}

# ============================================
# Date range
# END_DATE now tracks today automatically instead of a fixed 2025-12-31,
# so re-running this later always pulls up through the present.
# ============================================
START_DATE = "20000101"
END_DATE = date.today().strftime("%Y%m%d")

# ============================================
# Weather parameters
# Original 8 left untouched. Three added at the end for forecasting:
#   - T2MDEW    : dew point @ 2m -- a direct moisture measure, a stronger
#                 precipitation predictor than relative humidity alone
#   - CLOUD_AMT : cloud amount (%) -- directly informative for near-term
#                 rain likelihood and next-day solar radiation
#   - WS50M     : wind speed @ 50m -- sits above surface friction, so it
#                 carries a bit more of the larger-scale flow signal than
#                 WS10M alone (still no substitute for real upper-air data,
#                 but the best proxy POWER offers)
# ============================================
parameters = [
    "PRECTOTCORR",
    "T2M",
    "T2M_MAX",
    "T2M_MIN",
    "RH2M",
    "WS10M",
    "PS",
    "ALLSKY_SFC_SW_DWN",
    "T2MDEW",
    "CLOUD_AMT",
    "WS50M"
]

parameters_str = ",".join(parameters)

# ============================================
# NASA POWER API
# ============================================
url = "https://power.larc.nasa.gov/api/temporal/daily/point"

# ============================================
# Download and save each district separately
# ============================================
for district, coords in locations.items():

    print(f"\nDownloading {district}...")

    params = {
        "parameters": parameters_str,
        "community": "AG",
        "longitude": coords["longitude"],
        "latitude": coords["latitude"],
        "start": START_DATE,
        "end": END_DATE,
        "format": "JSON",
        "time-standard": "LST"
    }

    response = requests.get(
        url,
        params=params,
        timeout=60
    )

    print("Status code:", response.status_code)

    response.raise_for_status()

    data = response.json()

    # ============================================
    # Convert to DataFrame
    # ============================================
    weather = pd.DataFrame(
        data["properties"]["parameter"]
    )

    weather.index.name = "DATE"
    weather.reset_index(inplace=True)

    weather["DATE"] = pd.to_datetime(
        weather["DATE"],
        format="%Y%m%d"
    )

    # NASA POWER uses -999 as a missing-value sentinel -- flag it as NaN
    # so it isn't silently used as a real reading (matters more now that
    # END_DATE reaches right up to the present, where a day or two of
    # data can lag behind processing).
    weather.replace(-999, pd.NA, inplace=True)

    # ============================================
    # Rename columns
    # Original 8 renames untouched; 3 new ones added for the new params.
    # ============================================
    weather.rename(columns={
        "PRECTOTCORR": "Rainfall_mm",
        "T2M": "Temperature_C",
        "T2M_MAX": "Temperature_Max_C",
        "T2M_MIN": "Temperature_Min_C",
        "RH2M": "Relative_Humidity_percent",
        "WS10M": "Wind_Speed_mps",
        "PS": "Surface_Pressure_kPa",
        "ALLSKY_SFC_SW_DWN": "Solar_Radiation",
        "T2MDEW": "Dew_Point_C",
        "CLOUD_AMT": "Cloud_Amount_percent",
        "WS50M": "Wind_Speed_50m_mps"
    }, inplace=True)

    # ============================================
    # Save separate CSV
    # ============================================
    filename = f"{district.lower()}_weather_2000_2025.csv"

    weather.to_csv(
        filename,
        index=False
    )

    print(f"Saved: {filename}")
    print(f"Rows: {len(weather)}")
    print(f"Columns: {len(weather.columns)}")

print("\nAll 5 district datasets downloaded successfully!")



Status code: 200
Saved: beed_weather_2000_2025.csv
Rows: 9757
Columns: 12

Status code: 200
Saved: latur_weather_2000_2025.csv
Rows: 9757
Columns: 12

Status code: 200
Saved: osmanabad_weather_2000_2025.csv
Rows: 9757
Columns: 12

Status code: 200
Saved: solapur_weather_2000_2025.csv
Rows: 9757
Columns: 12

Status code: 200
Saved: ahmednagar_weather_2000_2025.csv
Rows: 9757
Columns: 12

All 5 district datasets downloaded successfully!


In [2]:
"""
Weather data download for the Maharashtra drought-district forecasting model.

Downloads three kinds of OBSERVED data (no forecasts are used anywhere):

  1. NASA POWER daily weather for the 5 target districts
  2. NASA POWER daily weather for regional "upstream" points around them.
     Monsoon rain systems usually form over the Bay of Bengal and travel
     west-northwest across central India, so weather changes at eastern points
     show up in these districts a few days later. Western points track the
     moisture-carrying monsoon winds from the Arabian Sea.
  3. Large-scale climate indices:
       - MJO (RMM1/RMM2, daily)  -> active/break phases of the monsoon (weeks 2-4)
       - ENSO (ONI, monthly)     -> El Nino / La Nina background state
       - IOD (DMI, monthly)      -> Indian Ocean Dipole background state

Re-running is cheap: existing files are updated incrementally (only the last
~60 days are re-downloaded, because the newest NASA POWER values are provisional).

Output layout:
  data/weather/districts/<name>_weather.csv
  data/weather/regional/<name>_weather.csv
  data/weather/locations.csv
  data/indices/mjo_rmm_daily.csv
  data/indices/enso_oni_monthly.csv
  data/indices/iod_dmi_monthly.csv
"""

import time
from datetime import date, timedelta
from pathlib import Path

import pandas as pd
import requests

# ============================================
# Paths and date range
# ============================================
OUT_DIR = Path("data")
DISTRICT_DIR = OUT_DIR / "weather" / "districts"
REGIONAL_DIR = OUT_DIR / "weather" / "regional"
INDEX_DIR = OUT_DIR / "indices"
for d in (DISTRICT_DIR, REGIONAL_DIR, INDEX_DIR):
    d.mkdir(parents=True, exist_ok=True)

START_DATE = date(2000, 1, 1)
END_DATE = date.today()
REFRESH_DAYS = 60  # re-download this many recent days on every run

# ============================================
# Target districts (what we forecast)
# ============================================
DISTRICTS = {
    "Beed":       {"latitude": 18.9891, "longitude": 75.7601},
    "Latur":      {"latitude": 18.4088, "longitude": 76.5604},
    "Osmanabad":  {"latitude": 18.1860, "longitude": 76.0419},
    "Solapur":    {"latitude": 17.6599, "longitude": 75.9064},
    "Ahmednagar": {"latitude": 19.0948, "longitude": 74.7480},
}

# ============================================
# Regional upstream points (predictors only)
# The 5 districts are close together, so one shared set serves all of them.
# ============================================
REGIONAL_POINTS = {
    # East / south-east: path of Bay of Bengal monsoon lows
    "Nanded":             {"latitude": 19.1383, "longitude": 77.3210, "direction": "east"},
    "Hyderabad":          {"latitude": 17.3850, "longitude": 78.4867, "direction": "southeast"},
    "Nagpur":             {"latitude": 21.1458, "longitude": 79.0882, "direction": "northeast"},
    "Raipur":             {"latitude": 21.2514, "longitude": 81.6296, "direction": "east"},
    "Visakhapatnam":      {"latitude": 17.6868, "longitude": 83.2185, "direction": "east"},
    "Bhubaneswar":        {"latitude": 20.2961, "longitude": 85.8245, "direction": "east"},
    "BayOfBengal_North":  {"latitude": 19.0000, "longitude": 88.0000, "direction": "east"},   # ocean point
    # West: monsoon winds and moisture from the Arabian Sea
    "Pune":               {"latitude": 18.5204, "longitude": 73.8567, "direction": "west"},
    "Mumbai":             {"latitude": 19.0760, "longitude": 72.8777, "direction": "west"},
    "ArabianSea_East":    {"latitude": 18.0000, "longitude": 71.0000, "direction": "west"},   # ocean point
}

# ============================================
# NASA POWER parameters
# Original 8 + your 3 additions kept. Added:
#   WD10M : wind direction @10m -- monsoon westerlies vs. break-phase shifts
#   WD50M : wind direction @50m -- larger-scale flow direction
#   QV2M  : specific humidity   -- absolute moisture, unlike RH it doesn't
#           swing with temperature during the day
# Regional points use a smaller set: only what helps predict rain arriving.
# ============================================
DISTRICT_PARAMS = {
    "PRECTOTCORR": "Rainfall_mm",
    "T2M": "Temperature_C",
    "T2M_MAX": "Temperature_Max_C",
    "T2M_MIN": "Temperature_Min_C",
    "RH2M": "Relative_Humidity_percent",
    "WS10M": "Wind_Speed_mps",
    "PS": "Surface_Pressure_kPa",
    "ALLSKY_SFC_SW_DWN": "Solar_Radiation",
    "T2MDEW": "Dew_Point_C",
    "CLOUD_AMT": "Cloud_Amount_percent",
    "WS50M": "Wind_Speed_50m_mps",
    "WD10M": "Wind_Direction_deg",
    "WD50M": "Wind_Direction_50m_deg",
    "QV2M": "Specific_Humidity_gkg",
}

REGIONAL_PARAMS = {
    "PRECTOTCORR": "Rainfall_mm",
    "T2M_MAX": "Temperature_Max_C",
    "RH2M": "Relative_Humidity_percent",
    "PS": "Surface_Pressure_kPa",
    "T2MDEW": "Dew_Point_C",
    "QV2M": "Specific_Humidity_gkg",
    "CLOUD_AMT": "Cloud_Amount_percent",
    "WS50M": "Wind_Speed_50m_mps",
    "WD50M": "Wind_Direction_50m_deg",
}

POWER_URL = "https://power.larc.nasa.gov/api/temporal/daily/point"


# ============================================
# Helpers
# ============================================
def get_with_retry(url, params=None, timeout=120, retries=4):
    """GET with exponential backoff (NASA POWER rate-limits and occasionally times out)."""
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            if r.status_code == 422:
                # Invalid request, usually a parameter name -- retrying won't help
                raise ValueError(f"Request rejected (422): {r.text[:500]}")
            r.raise_for_status()
            return r
        except (requests.RequestException,) as e:
            if attempt == retries:
                raise
            wait = 5 * 2 ** (attempt - 1)
            print(f"  attempt {attempt} failed ({e}); retrying in {wait}s")
            time.sleep(wait)


def fetch_power(lat, lon, param_map, start, end):
    """Download one point from NASA POWER and return a tidy daily DataFrame."""
    params = {
        "parameters": ",".join(param_map),
        "community": "AG",
        "latitude": lat,
        "longitude": lon,
        "start": start.strftime("%Y%m%d"),
        "end": end.strftime("%Y%m%d"),
        "format": "JSON",
        "time-standard": "LST",
    }
    data = get_with_retry(POWER_URL, params).json()

    df = pd.DataFrame(data["properties"]["parameter"])
    df.index = pd.to_datetime(df.index, format="%Y%m%d")
    df.index.name = "DATE"
    df = df.reset_index()

    # -999 is NASA POWER's missing-value sentinel
    df = df.replace(-999, pd.NA)
    df = df.rename(columns=param_map)

    # Drop trailing days that NASA POWER hasn't processed yet (all values missing)
    value_cols = [c for c in df.columns if c != "DATE"]
    df = df.dropna(subset=value_cols, how="all")
    return df


def update_point(name, coords, param_map, out_dir):
    """Create the file, or incrementally update it if it already exists."""
    path = out_dir / f"{name.lower()}_weather.csv"
    old = None
    start = START_DATE

    if path.exists():
        old = pd.read_csv(path, parse_dates=["DATE"])
        missing_cols = set(param_map.values()) - set(old.columns)
        if missing_cols:
            print(f"  new columns {sorted(missing_cols)} -> full re-download")
            old = None
        else:
            start = max(START_DATE, old["DATE"].max().date() - timedelta(days=REFRESH_DAYS))

    print(f"  downloading {start} -> {END_DATE}")
    new = fetch_power(coords["latitude"], coords["longitude"], param_map, start, END_DATE)

    if old is not None:
        # New download wins for overlapping dates (replaces provisional values)
        combined = pd.concat([old[old["DATE"] < pd.Timestamp(start)], new], ignore_index=True)
    else:
        combined = new

    combined = combined.drop_duplicates("DATE", keep="last").sort_values("DATE")
    combined.to_csv(path, index=False)

    # Quick data-quality report
    expected = pd.date_range(combined["DATE"].min(), combined["DATE"].max(), freq="D")
    gaps = len(expected) - len(combined)
    n_missing = int(combined.drop(columns="DATE").isna().sum().sum())
    print(f"  saved {path}  rows={len(combined)}  last={combined['DATE'].max().date()}  "
          f"missing_dates={gaps}  missing_values={n_missing}")
    time.sleep(2)  # be polite to the API


# ============================================
# Climate indices
# NOTE for feature building: monthly ONI/DMI are published after the month ends,
# so when predicting on a given day use the PREVIOUS month's value to avoid
# using information that wasn't available yet. MJO RMM is available with ~1 day lag.
# ============================================
def download_mjo():
    url = "http://www.bom.gov.au/climate/mjo/graphics/rmm.74toRealtime.txt"
    text = get_with_retry(url).text
    rows = []
    for line in text.splitlines():
        parts = line.split()
        if len(parts) < 7 or not parts[0].isdigit():
            continue  # header lines
        y, m, d = int(parts[0]), int(parts[1]), int(parts[2])
        rmm1, rmm2, phase, amp = float(parts[3]), float(parts[4]), int(parts[5]), float(parts[6])
        if abs(rmm1) > 100 or abs(rmm2) > 100:  # missing values are stored as 1.E36 / 999
            continue
        rows.append((pd.Timestamp(y, m, d), rmm1, rmm2, phase, amp))
    df = pd.DataFrame(rows, columns=["DATE", "RMM1", "RMM2", "MJO_Phase", "MJO_Amplitude"])
    df = df[df["DATE"] >= pd.Timestamp(START_DATE)]
    df.to_csv(INDEX_DIR / "mjo_rmm_daily.csv", index=False)
    print(f"  saved MJO: {len(df)} days, last={df['DATE'].max().date()}")


def download_oni():
    url = "https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt"
    text = get_with_retry(url).text
    season_to_month = {s: i + 1 for i, s in enumerate(
        ["DJF", "JFM", "FMA", "MAM", "AMJ", "MJJ", "JJA", "JAS", "ASO", "SON", "OND", "NDJ"])}
    rows = []
    for line in text.splitlines():
        parts = line.split()
        if len(parts) == 4 and parts[0] in season_to_month:
            # ONI for a 3-month season is labelled by its centre month
            rows.append((int(parts[1]), season_to_month[parts[0]], float(parts[3])))
    df = pd.DataFrame(rows, columns=["Year", "Month", "ONI"])
    df = df[df["Year"] >= START_DATE.year]
    df.to_csv(INDEX_DIR / "enso_oni_monthly.csv", index=False)
    print(f"  saved ONI: {len(df)} months, last={df['Year'].iloc[-1]}-{df['Month'].iloc[-1]:02d}")


def download_dmi():
    url = "https://psl.noaa.gov/gcos_wgsp/Timeseries/Data/dmi.had.long.data"
    text = get_with_retry(url).text
    rows = []
    for line in text.splitlines():
        parts = line.split()
        if len(parts) != 13 or not parts[0].isdigit():
            continue  # header/footer lines
        year = int(parts[0])
        for month, val in enumerate(parts[1:], start=1):
            v = float(val)
            if v > -99:  # -9999 = missing
                rows.append((year, month, v))
    df = pd.DataFrame(rows, columns=["Year", "Month", "DMI"])
    df = df[df["Year"] >= START_DATE.year]
    df.to_csv(INDEX_DIR / "iod_dmi_monthly.csv", index=False)
    print(f"  saved DMI: {len(df)} months, last={df['Year'].iloc[-1]}-{df['Month'].iloc[-1]:02d}")


# ============================================
# Main
# ============================================
def main():
    failures = []

    # Location metadata (useful later for distance/direction features)
    meta = [{"name": n, "role": "district", **c} for n, c in DISTRICTS.items()]
    meta += [{"name": n, "role": "regional", **c} for n, c in REGIONAL_POINTS.items()]
    pd.DataFrame(meta).to_csv(OUT_DIR / "weather" / "locations.csv", index=False)

    print("=== Target districts ===")
    for name, coords in DISTRICTS.items():
        print(f"\n{name}")
        try:
            update_point(name, coords, DISTRICT_PARAMS, DISTRICT_DIR)
        except Exception as e:
            print(f"  FAILED: {e}")
            failures.append(name)

    print("\n=== Regional upstream points ===")
    for name, coords in REGIONAL_POINTS.items():
        print(f"\n{name} ({coords['direction']})")
        try:
            update_point(name, coords, REGIONAL_PARAMS, REGIONAL_DIR)
        except Exception as e:
            print(f"  FAILED: {e}")
            failures.append(name)

    print("\n=== Climate indices ===")
    for label, fn in [("MJO", download_mjo), ("ONI", download_oni), ("DMI", download_dmi)]:
        print(f"\n{label}")
        try:
            fn()
        except Exception as e:
            print(f"  FAILED: {e}")
            failures.append(label)

    print("\n" + ("All downloads completed successfully!" if not failures
                  else f"Finished with failures: {failures} -- re-run to retry them."))


if __name__ == "__main__":
    main()

=== Target districts ===

Beed
  downloading 2000-01-01 -> 2026-09-17
  saved data\weather\districts\beed_weather.csv  rows=9754  last=2026-09-14  missing_dates=0  missing_values=109

Latur
  downloading 2000-01-01 -> 2026-09-17
  saved data\weather\districts\latur_weather.csv  rows=9754  last=2026-09-14  missing_dates=0  missing_values=109

Osmanabad
  downloading 2000-01-01 -> 2026-09-17
  saved data\weather\districts\osmanabad_weather.csv  rows=9754  last=2026-09-14  missing_dates=0  missing_values=109

Solapur
  downloading 2000-01-01 -> 2026-09-17
  saved data\weather\districts\solapur_weather.csv  rows=9754  last=2026-09-14  missing_dates=0  missing_values=109

Ahmednagar
  downloading 2000-01-01 -> 2026-09-17
  saved data\weather\districts\ahmednagar_weather.csv  rows=9754  last=2026-09-14  missing_dates=0  missing_values=109

=== Regional upstream points ===

Nanded (east)
  downloading 2000-01-01 -> 2026-09-17
  saved data\weather\regional\nanded_weather.csv  rows=9754  last=2